# Module 8 Practical: Predictive Modelling with PySpark

In this tutorial, we will use the **Boston House-price dataset** to build a predictive model.

The activity asks us to:

1. Compute correlations between the variables.
2. identify the three variables most strongly related to house price.
3. Create a degree-two polynomial regression model.
4. Use 70% of the data for training and 30% for testing.
5. Evaluate the model using R-squared.

The target variable is **MEDV**, which represents the median house value.


## 1. Import the required libraries

The libraries below have different purposes:

- `pandas` helps us display and inspect the correlation matrix.
- `seaborn` and `matplotlib` create the correlation heatmap.
- `SparkSession` starts PySpark.
- `VectorAssembler` combines several input columns into one feature vector.
- `PolynomialExpansion` creates degree-two terms.
- `LinearRegression` builds the regression model.
- `RegressionEvaluator` calculates the test R-squared value.


In [ ]:
# Install seaborn if it is not already available
!pip install seaborn gdown -q

import pandas as pd
import gdown
import seaborn as sb
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, PolynomialExpansion
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


## 2. Start Spark

`SparkSession.builder.getOrCreate()` starts a new Spark session or reuses an existing one.


In [ ]:
spark = SparkSession.builder.getOrCreate()


## 3. Download the dataset

The dataset is stored in Google Drive. `gdown.download()` downloads it and saves it as `boston.csv` in the notebook's current folder.


In [ ]:
file_id = "1RyFqFEUYp9o4b4pSwwDR_YD1s6aBTCHF"
gdown.download(id=file_id, output="boston.csv", quiet=False)


## 4. Import the Boston housing dataset

Place `boston.csv` in the same folder as this notebook.

Important variables include:

- `CRIM`: crime rate
- `RM`: average number of rooms
- `PTRATIO`: pupil–teacher ratio
- `LSTAT`: percentage of the population with lower socioeconomic status
- `MEDV`: median house value, which is the variable we want to predict

`inferSchema=True` asks Spark to recognise numeric columns as numbers instead of text.


In [ ]:
boston_housing = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("boston.csv")
)

boston_housing.show(5)
boston_housing.printSchema()


## 5. Compute the correlation matrix

Correlation measures the strength and direction of a relationship between two numeric variables.

- A value close to `1` indicates a strong positive relationship.
- A value close to `-1` indicates a strong negative relationship.
- A value close to `0` indicates a weak linear relationship.

We first convert the Spark DataFrame to a pandas DataFrame because pandas and seaborn make the correlation matrix easy to display.


In [ ]:
boston_pd = boston_housing.toPandas()

correlation_matrix = boston_pd.corr(numeric_only=True)

correlation_matrix.round(2)


The heatmap provides a visual version of the same correlation matrix. Darker or stronger colours indicate stronger relationships.


In [ ]:
plt.figure(figsize=(12, 9))

sb.heatmap(
    correlation_matrix,
    cmap="Blues",
    annot=True,
    fmt=".2f"
)

plt.title("Boston Housing Correlation Matrix")
plt.show()


## 6. Select the top three variables

We are interested in variables that have the strongest relationship with `MEDV`.

The code below:

1. selects the `MEDV` column from the correlation matrix;
2. removes the correlation of `MEDV` with itself;
3. converts negative values to positive values using `abs()`;
4. sorts the correlations from strongest to weakest; and
5. displays the top three variables.


In [ ]:
correlations_with_medv = (
    correlation_matrix["MEDV"]
    .drop("MEDV")
    .abs()
    .sort_values(ascending=False)
)

top_three_variables = correlations_with_medv.head(3)

print(top_three_variables)


For the Boston housing dataset, the three strongest variables are normally:

- `LSTAT`
- `RM`
- `PTRATIO`

We will use these three variables to build the model.


In [ ]:
selected_variables = ["LSTAT", "RM", "PTRATIO"]


## 7. Prepare the three input variables

PySpark machine-learning models expect all predictor variables to be stored in one vector column.

`VectorAssembler` combines `LSTAT`, `RM` and `PTRATIO` into a new column called `features`.


In [ ]:
assembler = VectorAssembler(
    inputCols=selected_variables,
    outputCol="features"
)

model_data = assembler.transform(boston_housing)

model_data.select("MEDV", "features").show(5, truncate=False)


## 8. Create degree-two polynomial features

A degree-two polynomial model can include:

- the original variables;
- squared variables, such as `LSTAT²`; and
- interactions between variables, such as `LSTAT × RM`.

`PolynomialExpansion(degree=2)` creates these terms automatically.


In [ ]:
polynomial = PolynomialExpansion(
    degree=2,
    inputCol="features",
    outputCol="polynomial_features"
)

polynomial_data = (
    polynomial
    .transform(model_data)
    .select("MEDV", "polynomial_features")
)

polynomial_data.show(5, truncate=False)


## 9. Split the data into training and test datasets

- The training dataset is used to build the model.
- The test dataset is kept separate and used to evaluate the model.

The `seed` makes the split reproducible, so the same rows are selected each time the notebook is run.


In [ ]:
training_data, test_data = polynomial_data.randomSplit(
    [0.7, 0.3],
    seed=42
)

print("Training rows:", training_data.count())
print("Test rows:", test_data.count())


## 10. Build the regression model

`LinearRegression` can fit a polynomial model because the polynomial terms have already been created in `polynomial_features`.

- `featuresCol` identifies the input feature column.
- `labelCol` identifies the target variable.


In [ ]:
regression = LinearRegression(
    featuresCol="polynomial_features",
    labelCol="MEDV"
)

regression_model = regression.fit(training_data)


## 11. Make predictions on the test data

`transform()` applies the trained model to the test dataset and creates a new column called `prediction`.


In [ ]:
predictions = regression_model.transform(test_data)

predictions.select(
    "MEDV",
    "prediction"
).show(10)


## 12. Compute the test R-squared value

R-squared measures how much of the variation in the target variable is explained by the model.

- A value closer to `1` generally indicates a better fit.
- A value near `0` indicates that the model explains little of the variation.
- R-squared should be interpreted together with the context of the problem and the quality of the data.


In [ ]:
evaluator = RegressionEvaluator(
    labelCol="MEDV",
    predictionCol="prediction",
    metricName="r2"
)

test_r_squared = evaluator.evaluate(predictions)

print("R-squared on the test data:", test_r_squared)


## 13. Discussion prompts

When writing your discussion forum response, consider:

- Which variables had the strongest correlation with `MEDV`?
- Was each relationship positive or negative?
- What R-squared value did the model achieve on the test data?
- Did the model explain a large or small proportion of the variation in house prices?
- What difficulties did you experience when importing the data, preparing features or interpreting the results?

## Learning Activity 2: Adaptability of models

A predictive model learns patterns from historical data. It may perform poorly when future data differs from the data used to train it.

For example, a house-price model trained in one city may not work well in another city because property types, incomes, transport systems and market conditions differ. Models therefore need to be checked, updated and sometimes retrained when they are applied to a new problem or environment.
